<a href="https://colab.research.google.com/github/Carinaaa/ML-Learning-Path/blob/intro-LLM/RAG_Intro_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:
import os
import glob
import gradio as gr
from openai import OpenAI
from google.colab import userdata
import requests
# imports for langchain
from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import CharacterTextSplitter

In [3]:
api_key = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = api_key # set it as an env var

open_ai = OpenAI()
model = 'gpt-4o-mini'

In [4]:
context = {}
all_employees = ['Alex Chen', 'Alex Harper', 'Alex Thomson', 'Avery Lancaster', 'Emily Carter', 'Emily Tran', 'Jordan Blake', 'Jordan K. Bishop', 'Maxine Thompson', 'Oliver Spencer', 'Samantha Greene', 'Samuel Trenton']
for e in all_employees:
  context[e] = requests.get(f'https://raw.githubusercontent.com/ed-donner/llm_engineering/refs/heads/main/week5/knowledge-base/employees/{e.replace(" ", "%20")}.md').text

employee_context = context.copy()
try:
  os.mkdir('knowedge-base/')
  os.mkdir('knowedge-base/employees')
except FileExistsError:
  print("Dirs already exists.")
for e in all_employees:
  with open(f'knowedge-base/employees/{e}.md', 'w') as f:
    f.write(employee_context[e])

In [5]:
all_products = ['Carllm', 'Homellm', 'Markellm', 'Rellm']
products_context = {}
for p in all_products:
  context[p] = requests.get(f'https://raw.githubusercontent.com/ed-donner/llm_engineering/refs/heads/main/week5/knowledge-base/products/{p}.md').text
  products_context[p] = context[p]

try:
  os.mkdir('knowedge-base/products')
except FileExistsError:
  print("Dirs already exists.")
for p in all_products:
  with open(f'knowedge-base/products/{p}.md', 'w') as f:
    f.write(products_context[p])

In [6]:
company_info = ['about', 'careers', 'overview']
company_context = {}
for e in company_info:
  context[e] = requests.get(f'https://raw.githubusercontent.com/ed-donner/llm_engineering/refs/heads/main/week5/knowledge-base/company/{e}.md').text
  company_context[e] = context[e]
try:
  os.mkdir('knowedge-base/company')
except FileExistsError:
  print("Dirs already exists.")
for e in company_info:
  with open(f'knowedge-base/company/{e}.md', 'w') as f:
    f.write(company_context[e])

In [7]:
contracts_info = ['Contract with Apex Reinsurance for Rellm', 'Contract with Belvedere Insurance for Markellm', 'Contract with BrightWay Solutions for Markellm',
                'Contract with EverGuard Insurance for Rellm', 'Contract with GreenField Holdings for Markellm', 'Contract with GreenValley Insurance for Homellm',
                'Contract with Greenstone Insurance for Homellm','Contract with Pinnacle Insurance Co. for Homellm', 'Contract with Roadway Insurance Inc. for Carllm',
                'Contract with Stellar Insurance Co. for Rellm', 'Contract with TechDrive Insurance for Carllm', 'Contract with Velocity Auto Solutions for Carllm']
contracts_context = {}
for e in contracts_info:
  context[e] = requests.get(f'https://raw.githubusercontent.com/ed-donner/llm_engineering/refs/heads/main/week5/knowledge-base/contracts/{e.replace(" ", "%20")}.md').text
  contracts_context[e] = context[e]
try:
  os.mkdir('knowedge-base/contracts')
except FileExistsError:
  print("Dirs already exists.")
for e in contracts_info:
  with open(f'knowedge-base/contracts/{e}.md', 'w') as f:
    f.write(contracts_context[e])

In [8]:
context.keys()

dict_keys(['Alex Chen', 'Alex Harper', 'Alex Thomson', 'Avery Lancaster', 'Emily Carter', 'Emily Tran', 'Jordan Blake', 'Jordan K. Bishop', 'Maxine Thompson', 'Oliver Spencer', 'Samantha Greene', 'Samuel Trenton', 'Carllm', 'Homellm', 'Markellm', 'Rellm', 'about', 'careers', 'overview', 'Contract with Apex Reinsurance for Rellm', 'Contract with Belvedere Insurance for Markellm', 'Contract with BrightWay Solutions for Markellm', 'Contract with EverGuard Insurance for Rellm', 'Contract with GreenField Holdings for Markellm', 'Contract with GreenValley Insurance for Homellm', 'Contract with Greenstone Insurance for Homellm', 'Contract with Pinnacle Insurance Co. for Homellm', 'Contract with Roadway Insurance Inc. for Carllm', 'Contract with Stellar Insurance Co. for Rellm', 'Contract with TechDrive Insurance for Carllm', 'Contract with Velocity Auto Solutions for Carllm'])

In [9]:
db_name = "vector_db"

In [10]:
folders = glob.glob("knowedge-base/*")
text_loader_kwargs = {'encoding': 'utf-8'}
documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs=text_loader_kwargs)
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

len(documents)

31

In [11]:
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

In [12]:
len(chunks)

123

In [13]:
chunks[6]

Document(metadata={'source': 'knowedge-base/products/Carllm.md', 'doc_type': 'products'}, page_content='- **Professional Tier**: $2,500/month\n  - For medium-sized companies.\n  - All Basic Tier features plus advanced analytics and fraud detection.\n\n- **Enterprise Tier**: $5,000/month\n  - Customized solutions for large insurance firms.\n  - Comprehensive support, full feature access, and integration with existing systems.\n\nContact our sales team for a personalized quote and discover how Carllm can transform your auto insurance offerings!\n\n## 2025-2026 Roadmap\n\nIn our commitment to continuous improvement and innovation, Insurellm has outlined the following roadmap for Carllm:\n\n### Q1 2025: Launch Feature Enhancements\n- **Expanded data integrations** for better risk assessment.\n- **Enhanced fraud detection algorithms** to reduce losses.\n\n### Q2 2025: Customer Experience Improvements\n- Launch of a new **mobile app** for end-users.\n- Introduction of **telematics-based pric

In [14]:
doc_types = set(chunk.metadata['doc_type'] for chunk in chunks)
print(f"Document types found: {', '.join(doc_types)}")

Document types found: company, products, contracts, employees


In [15]:
for chunk in chunks:
    if 'CEO' in chunk.page_content:
        print(chunk)
        print("_________")

page_content='# Avery Lancaster

## Summary
- **Date of Birth**: March 15, 1985  
- **Job Title**: Co-Founder & Chief Executive Officer (CEO)  
- **Location**: San Francisco, California  

## Insurellm Career Progression
- **2015 - Present**: Co-Founder & CEO  
  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  

- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  
  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the tech sector.' metadata={'source': 'knowedge-base/employees/Avery Lancaster.md', 'doc_type': 'employees'}
_________
page_content='3. **Regular Updates:** Insurellm will offer ongoing updat